# OCR a ticker's filings

`ENVIRONMENT` is the only switch: `"LOCAL"` parses on this machine's card, `"KAGGLE"` ships the
same job to a Kaggle T4 through `kgpu`. Edit **1 · Parameters**, then run top to bottom.

⚠️ **The T4 is not faster than this laptop** — 0.62-0.68 s/page here against 0.78-0.95 there,
plus a ~5 min queue and the payload upload. What it buys is *a second machine running in
parallel, free*: worth it for a 70-document ticker, worth nothing for one quarter.

⚠️ **"THE RUN FINISHED" AND "THE CSV CHANGED" ARE DIFFERENT FACTS.** A green run can write no
statement CSV at all — every statement refused for an empty `sane` band (`BND-1`). Sections
**7-9** answer it in widening order of trust: the run folder → refused vs written → the CSVs.

**An interrupted LOCAL run keeps every quarter that finished** — each is upserted the moment it
lands, and the three CSVs are backed up before the first write. ⚠️ On KAGGLE that guarantee is
the PULL's, not the run's: a kernel writes `/kaggle/working` and has no path to this disk.

⚠️ **Nothing from a non-bank template may be quoted as a fundamental yet** — `CRP-1`.

The measurements behind these lines: [`kgpu/PDF_OCR.md`](kgpu/PDF_OCR.md), `docs/ISSUES.md`,
CLAUDE.md §6-2.


## 1 · Parameters — the only cell you edit

Everything below this cell is machinery.


In [1]:
# ── PARAMETERS — the only cell you edit ───────────────────────────────────────
ENVIRONMENT = "LOCAL"        # "LOCAL" = parse here | "KAGGLE" = ship it to a T4
EXCHANGE    = "HOSE"         # HOSE | HNX | UPCOM
SYMBOL      = "TCB"          # ticker, as CafeF files it

# WHICH QUARTERS — YYYY-QQ. "2026-Q4" and the zero-padded "2026-04" are the same quarter.
#   []  or  None  ->  EVERY quarter this ticker files   (⚠️ 70+ documents, hours)
# ⚠️ The repo-native "Q3-2014" is REFUSED rather than quietly accepted: a typo has to report
#    itself as a typo, not as a quarter CafeF does not file.
QUARTERS = []        # ["2011-03"] is the same quarter, written the other way

# ⚠️ What to do about a quarter ALREADY on disk:
#   False -> FILL THE GAPS. One reading `pdf` in all three statements is dropped before any OCR
#            (and before it is uploaded); a figure that DIFFERS is never written over it.
#   True  -> re-parse every selected quarter and let the result replace what disk holds.
#   ⚠️ To replace ONE wrong row use REPAIR below, never this — see the note there.
OVERWRITE = False

# UPSERT the accepted statements into raw_data/.../statements/*.csv, through `pdf_ocr_merge`:
# it BACKS THE THREE CSVs UP FIRST, prints every changed cell, and refuses four things it
# cannot judge — a cumulative income statement whose missing quarters WERE filed (one whose
# priors never existed is written instead, carrying `months=6`/`12`), a statement whose `sane`
# band was empty, a figure that DIFFERS from a good `pdf` row, and ⚠️ a document any of whose
# layers RAISED (`VCR-1`: an exception measures the MACHINE, not the filing, so whatever won
# the cascade won by default).
#   LOCAL  -> one quarter at a time, as each finishes. This is the interruption guarantee.
#   KAGGLE -> once, after the pull. A kernel has no path to this disk.
MERGE_INTO_CSV = True

# ⚠️ BOOTSTRAP A TICKER THAT HAS NO STATEMENT CSV YET — and it lifts a real guard. `sane`'s
#    magnitude band is rebuilt from the `pdf` rows ALREADY ON DISK (`seed_history`), so a first
#    run has none, `sane` fails open, the merge then refuses every statement, nothing is
#    written, and the band is still empty next time — the loop closes on itself (`BND-1`).
#   True  -> write those statements anyway. The ONLY way a new ticker is bootstrapped, and it
#            lifts NO other refusal.
#   False -> keep the guard. Correct for a ticker that already has history on disk.
# ⚠️ What it costs is the guard: screen the artefact — the unit per report, and total assets
#    quarter on quarter — before quoting a bootstrap run. Both read off the run folder alone:
#    no PDF, no OCR, no network.
FORCE_EMPTY_BAND = True

TEMPLATE     = None      # None = RESOLVE it (templates.csv, then CafeF's fingerprint). ⚠️ Never defaulted to "bank".
ALLOW_PARENT = False     # fall back to the STANDALONE filing where no consolidated one exists
PERIODS      = None      # the repo-native form, e.g. ["Q3-2014"]. Optional, and INTERSECTS with QUARTERS.
LAYERS       = None      # None = the full cascade, in cascade order
COMPARE      = True      # score every parsed cell against the statement CSV already on disk
NOTES        = ""        # free text into the run folder; blank writes a sensible default

# ⚠️ REPAIR — REPLACE A `pdf` ROW THAT IS ALREADY ON DISK AND WRONG. Name the exact
#    (quarter, statement) pairs; anything not named keeps the DIFFERS refusal. The quarter is
#    the REPO-NATIVE form here:   REPAIR = [("Q3-2009", "income_statement")]
# ⚠️ `OVERWRITE = True` IS THE WRONG TOOL FOR THIS, AND THE REASON IS MEASURED. It lifts DIFFERS
#    for every statement of every quarter in the run — and a `pdf_ocr_job` run is NOT the run
#    that wrote those rows: its `sane` band is rebuilt from disk where a full `build()`
#    accumulates one as it goes, so the two escalate DIFFERENTLY and the seeded run can win on
#    an EARLIER, POORER layer. On ACB 2026-08-30 it would have replaced a 33-item balance sheet
#    with a 19-item one while repairing another statement, reporting only "DIFFERS in N columns".
# ⚠️ Read the DIFFERS report in section 7 first, and decide against the FILING (a printed
#    subtotal, the next quarter's comparative column) — never by preferring the newer run.
REPAIR = []
REPAIR_APPLY = False     # False = print the plan and change nothing. True once you agree.

EXECUTE  = True          # False = resolve and print the plan, spend nothing
REHEARSE = True          # KAGGLE only: the worker side, locally, no quota (~60 s)


## 2 · Setup — validate the parameters, find the repo


In [2]:
# ── SETUP — validate the parameters and find the repo ─────────────────────────
# ⚠️ Checked HERE, before a payload is built or a page is rendered: every one of these is a
# mistake that would otherwise surface hours later, or as a spent Kaggle round trip.
import os
import sys
from pathlib import Path

ENVIRONMENT = str(ENVIRONMENT).upper()
EXCHANGE = str(EXCHANGE).upper()
SYMBOL = str(SYMBOL).upper()
if ENVIRONMENT not in ("LOCAL", "KAGGLE"):
    raise ValueError(f"ENVIRONMENT must be 'LOCAL' or 'KAGGLE', not {ENVIRONMENT!r}")
if EXCHANGE not in ("HOSE", "HNX", "UPCOM"):
    raise ValueError(f"EXCHANGE must be HOSE, HNX or UPCOM, not {EXCHANGE!r}")

REPO = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "src" / "kaggle_gpu").is_dir()), None)
if REPO is None:
    raise RuntimeError(f"no src/kaggle_gpu at or above {Path.cwd()} — open this notebook "
                       f"from inside the repo.")
# ⚠️ `kgpu` stages the payload and talks to the Kaggle client relative to the CWD, so the
# notebook anchors itself the way a shell would. LOCAL does not need it and gets it anyway:
# one behaviour, printed, beats two that differ by a mode.
os.chdir(REPO / "src" / "kaggle_gpu")
for _p in (REPO / "src", REPO / "src" / "kaggle_gpu"):
    if str(_p) not in sys.path:
        sys.path.insert(0, str(_p))

# ⚠️ A LONG-LIVED KERNEL PINS THE REPO TO THE COMMIT IT FIRST IMPORTED — `import` is a no-op
# once a module is in `sys.modules`, so re-running this notebook after the repo moves underneath
# it runs the OLD code. The loud form is an AttributeError; ⚠️ the silent form is an OCR run
# executing a previous commit's parser while `metadata.json` records HEAD's hash — a run folder
# that names code it did not run. So the repo's OWN packages are dropped here and re-imported
# from disk on every pass; third-party ones (torch, onnxruntime) are left alone, they do not
# move. ⚠️ It re-imports, so run this notebook TOP TO BOTTOM.
_OURS = ("kgpu", "utils", "web_scraper")
_RELOADED = [_n for _n in list(sys.modules) if _n.split(".")[0] in _OURS]
for _n in _RELOADED:
    del sys.modules[_n]

from utils import progress                          # noqa: E402
from web_scraper import pdf_ocr_job as job          # noqa: E402

# ⚠️ FOLDED ONCE, HERE. "2026-04" and "2026-Q4" are one quarter, and the job name, the payload
# directory and the Kaggle kernel slug are all derived from this list — two spellings that
# reached those would be two runs racing for one slug. `None` means every quarter.
QUARTERS = job.canonical_quarters(QUARTERS)

# The task label every progress line carries. ONE string, built once: LOCAL replaces it per
# document (`doc 2/3 HOSE_TCB Q3-2013`), KAGGLE keeps it for all six steps.
LABEL = f"{EXCHANGE}_{SYMBOL} " + (" ".join(QUARTERS) if QUARTERS else "all quarters")

print(f"environment : {ENVIRONMENT}")
print(f"ticker      : {EXCHANGE}_{SYMBOL}")
print(f"quarters    : {QUARTERS or 'ALL — every quarter this ticker files'}")
print(f"overwrite   : {OVERWRITE}"
      + ("" if OVERWRITE else "   (quarters already `pdf` in all three are skipped)"))
print(f"upsert csv  : {MERGE_INTO_CSV}"
      + ("   per quarter, as each finishes" if MERGE_INTO_CSV and ENVIRONMENT == "LOCAL"
         else "   after the pull" if MERGE_INTO_CSV else ""))
print(f"bootstrap   : {FORCE_EMPTY_BAND}"
      + ("   an EMPTY `sane` band is written anyway — the only way a new ticker "
         "starts" if FORCE_EMPTY_BAND else "   an EMPTY `sane` band is REFUSED"))
print(f"repo        : {REPO}")
print(f"cwd         : {Path.cwd()}")
print(f"code        : {REPO / 'src'}"
      + (f"   ({len(_RELOADED)} cached module(s) dropped, re-imported from disk)"
         if _RELOADED else "   (first import in this kernel)"))
# ⚠️ The percentage is a POSITION IN THE PLAN and not a fraction of the time left — a filing
# accepted at layer 1 of 47 costs ~1 min and one that defeats the cascade cost 33. Said here,
# once, because it is on every line below it.
print(f"log shape   : {progress.format_line(0.337, 'task', 'sub-task', 'detail')}"
      f"   ← overall %, a position in the plan")


environment : LOCAL
ticker      : HOSE_TCB
quarters    : ALL — every quarter this ticker files
overwrite   : False   (quarters already `pdf` in all three are skipped)
upsert csv  : True   per quarter, as each finishes
bootstrap   : True   an EMPTY `sane` band is written anyway — the only way a new ticker starts
repo        : d:\GIT\master-thesis
cwd         : d:\GIT\master-thesis\src\kaggle_gpu
code        : d:\GIT\master-thesis\src   (first import in this kernel)
log shape   :  33.7% - task - sub-task - detail   ← overall %, a position in the plan


## 3 · The plan — what would run, before anything is spent


In [3]:
# ── THE JOB — resolved and printed, before anything is spent ──────────────────
# ⚠️ Both branches end at the SAME object. `pdf_ocr.job()` writes a `JobSpec`'s fields into the
# worker notebook's parameter cell, and the worker builds the JobSpec from them — so a LOCAL run
# and a KAGGLE run of the same parameters are one procedure on two machines, not two. What
# differs is the stack, and every run records its `stack_fingerprint`.
SPEC = CFG = PREPARED = None

if ENVIRONMENT == "LOCAL":
    SPEC = job.JobSpec(
        exchange=EXCHANGE, symbol=SYMBOL, periods=PERIODS, quarters=QUARTERS,
        allow_parent=ALLOW_PARENT, overwrite=OVERWRITE, template=TEMPLATE, layers=LAYERS,
        compare_with_disk=COMPARE, merge_into_csv=MERGE_INTO_CSV,
        force_empty_band=FORCE_EMPTY_BAND,
        notes=NOTES or f"ENVIRONMENT=LOCAL overwrite={OVERWRITE}",
    )
    # ⚠️ `prepare()` resolves the data root, the models, the TEMPLATE and the document list and
    # RAISES on any of them — no OCR, no PDF. It also raises, in as many words, when every
    # quarter you asked for is already parsed and OVERWRITE is False.
    PREPARED = SPEC.prepare()
    print("\n".join(PREPARED.describe()))
    print()
    for _t in PREPARED.tasks:
        print(f"  {_t.period:<8} {_t.file[:56]:<56} "
              f"{os.path.getsize(_t.path) / 1024 ** 2:>6.1f} MB"
              + ("  CUMULATIVE" if _t.cumulative else ""))
    # ⚠️ THE CEILING, BEFORE ANY OF IT IS SPENT. The bill is `pages x OCR passes`, and the 49
    # layers are only 7 passes — a layer that changes only the mapping or a gate re-maps a parse
    # the page cache already holds. Both numbers are free: `page_count` opens the PDF without
    # rendering a pixel, and the pass count is a property of the cascade.
    # ⚠️ It is a CEILING, loose in the honest direction: `scan` stops as soon as all three
    # statements are behind it (BID Q3-2011 reads 7 pages of 32) and the cascade stops at the
    # first layer that accepts. What it tells you is which filing would be dear IF something in
    # it cannot be read — that is the only case that pays it.
    import fitz                                       # noqa: E402
    from web_scraper.cafef_financials import ocr_key  # noqa: E402

    PASSES = len({ocr_key(_l) for _l in PREPARED.layers})
    PAGES = 0
    for _t in PREPARED.tasks:
        try:
            with fitz.open(_t.path) as _d:
                PAGES += _d.page_count
        except Exception as _e:                       # a damaged page tree is `scan`'s problem
            print(f"  ⚠️ could not count pages of {_t.file}: {_e}")
    print("")
    print(f"  ceiling      : {PAGES} page(s) x {PASSES} OCR pass(es) = "
          f"{PAGES * PASSES:,} page-reads at most")
    print(f"                 ~{PAGES * PASSES * 0.65 / 60:.0f} min at 0.65 s/page "
          f"(onnx@200 on this laptop; the 300/400 dpi passes cost more).")
    print("                 A filing accepted at layer 1 pays ONE pass over the pages "
          "up to its last")
    print("                 statement, which is the usual case — see the run log.")

    if PREPARED.template != "bank":
        print(f"\n⚠️ CRP-1: this is a `{PREPARED.template}` filing. `C_LIABILITIES` still "
              f"misses on corp,\n   so the balance sheet reconciles on the TRIVIAL "
              f"`assets == resources` — true by\n   construction on any page that reads both. "
              f"Nothing from a non-bank run may be\n   quoted as a fundamental yet.")
else:
    from kgpu import pdf_ocr, runner                 # noqa: E402

    CFG = pdf_ocr.job(
        SYMBOL, exchange=EXCHANGE, periods=PERIODS, quarters=QUARTERS,
        allow_parent=ALLOW_PARENT, overwrite=OVERWRITE, template=TEMPLATE, layers=LAYERS,
        compare=COMPARE, notes=NOTES, merge_statements=MERGE_INTO_CSV,
        # ⚠️ NOT a worker parameter. The worker cannot upsert — it writes /kaggle/working and
        # exits — so this is the PULL's knob, read by `runner.merge_statements` on this machine.
        force_empty_band=FORCE_EMPTY_BAND,
    )
    print("\n".join(pdf_ocr.describe(CFG)))
    print()
    # The filings this selects are the filings the WORKER will open: `plan()` runs HERE, so the
    # payload cannot diverge from the worker's own choice.
    runner.plan(CFG)


symbol       : HOSE_TCB
template     : bank   (detect_template (CafeF fingerprint, over the network))
documents    : 9  (Q4-2009, Q4-2010, Q2-2012, Q1-2013, Q1-2017, Q3-2017, Q2-2019, Q3-2019 …)
quarters     : all   selected ['2009-Q4', '2010-Q4', '2012-Q2', '2013-Q1', '2017-Q1', '2017-Q3', '2019-Q2', '2019-Q3', '2021-Q1']
skipped      : 50 quarter(s) already `pdf` in all three statements (2011-Q4, 2012-Q4, 2013-Q2, 2013-Q3, 2013-Q4, 2014-Q1, 2014-Q2, 2014-Q3 …)
cascade      : 55 of 55 layers
data root    : D:\GIT\master-thesis\raw_data\cafef
models       : det=deepdoc_det.onnx vietocr=vgg_seq2seq.pth
vietocr cfg  : D:\GIT\master-thesis\src\web_scraper\models\vietocr_vgg_seq2seq.yml

  Q4-2009  FY-2009_bao_cao_tai_chinh_hop_nhat_nam_2009_da_kiem_toan    0.5 MB  CUMULATIVE
  Q4-2010  FY-2010_bao_cao_tai_chinh_hop_nhat_nam_2010_da_kiem_toan    2.4 MB  CUMULATIVE
  Q2-2012  Q2-2012_bao_cao_tai_chinh_hop_nhat_quy_2_nam_2012.pdf      11.3 MB
  Q1-2013  Q1-2013_bao_cao_tai_chinh_hop_nhat_quy

## 4 · Settled absences — what a re-run cannot change

A cascade prints the same word — `absent` — whether the filing **does not contain** a statement
or every layer refused the one it found. The first kind is permanent: `missing` is then the
correct answer (§5 rule 24), and no layer, engine or DPI can change it. The cell reads that
reason out of previous run folders, and answers only from recorded ones — a run older than
artefact **schema v4** carries none, which is silence and not a clean bill (§5 rule 2).
⚠️ It reports; it does not filter: a filing can be re-uploaded.


In [4]:
# ── SETTLED ABSENCES — what a re-run cannot change ────────────────────────
# ⚠️ THE LOOP THIS BREAKS WAS REAL. ACB's Q2-2009 and Q3-2009 cash flows were put through all 50
# layers FOUR times on 2026-08-30. Both filings are three-page `BÁO CÁO TÀI CHÍNH TÓM TẮT` forms
# (Mẫu CBTT-03) carrying a condensed balance sheet and a four-line P&L — there is no cash flow
# statement in the document, so no run could ever have produced one. Nothing in the artefact
# said so until schema v4.
SETTLED = job.settled_absences(REPO / "reports" / "pdf_ocr", EXCHANGE, SYMBOL)

# Only the quarters THIS run would open. ⚠️ `SETTLED` is keyed `YYYY-QQ` — the sortable form
# QUARTERS is written in — while a task carries the repo-native `QQ-YYYY`. `as_quarter` is the
# converter; comparing the two spellings directly matches NOTHING and looks exactly like
# "nothing is settled".
_asked = ([job.as_quarter(t.period) for t in PREPARED.tasks] if PREPARED is not None
          else [job.normalise_quarter(q) for q in (QUARTERS or [])])
_hits = {p: SETTLED[p] for p in _asked if p in SETTLED} if _asked else dict(SETTLED)

if not _hits:
    print(f"no settled absence recorded for {EXCHANGE}_{SYMBOL} among the selected quarters.")
    print("⚠️ That is NOT 'everything is parseable': a run older than artefact schema v4")
    print("   recorded no reason at all, so this is SILENCE, not a clean bill.")
else:
    print("⚠️ ALREADY SETTLED — the filing does not CONTAIN these statements.")
    print("   A re-run cannot change them, at any layer, engine or DPI:")
    print("")
    for _period in sorted(_hits):
        for _report, _run in sorted(_hits[_period].items()):
            print(f"   {_period:<9} {_report:<18} measured by  {_run}")
    print("")
    print("   `missing` is the correct and permanent answer for the rows above")
    print("   (CLAUDE.md §5 rule 24: a quarter no readable PDF can produce is `missing`).")
    print("   Re-running them costs the full cascade and returns the same word. Drop them")
    print("   from QUARTERS, or keep them only to re-parse the OTHER statements of the")
    print("   same filing.")


⚠️ ALREADY SETTLED — the filing does not CONTAIN these statements.
   A re-run cannot change them, at any layer, engine or DPI:

   2009-Q4   cash_flow          measured by  20260902-030340__hose_tcb__pdf_ocr
   2010-Q4   cash_flow          measured by  20260902-030340__hose_tcb__pdf_ocr
   2017-Q1   cash_flow          measured by  20260901-204534__hose_tcb__pdf_ocr
   2017-Q3   cash_flow          measured by  20260901-204534__hose_tcb__pdf_ocr

   `missing` is the correct and permanent answer for the rows above
   (CLAUDE.md §5 rule 24: a quarter no readable PDF can produce is `missing`).
   Re-running them costs the full cascade and returns the same word. Drop them
   from QUARTERS, or keep them only to re-parse the OTHER statements of the
   same filing.


## 5 · Rehearse — KAGGLE only: the worker side, locally, no quota


In [5]:
# ── STAGE + REHEARSE — KAGGLE only: the worker side, locally, no quota ────────
# ⚠️ THE PAYLOAD IS STAGED HERE, AND IT HAS TO BE: a rehearsal runs the worker against
# `.payload/<job>/`, so there is nothing to rehearse until that exists. `export` is local and
# free — it writes the zip, it does not upload; the RUN cell below re-exports and uploads, so
# nothing here commits you to anything.
# ⚠️ The rehearsal runs no OCR pass. What it proves is that the payload holds every input the
# parse reads, under BOTH of Kaggle's mount layouts, and it prints the magnitude band `sane`
# will get. AN EMPTY BAND IS THE WARNING TO STOP FOR: `sane` fails open without one, and that
# is the documented way a run writes a wrong figure (CLAUDE.md §6-2-octodecies).
if ENVIRONMENT == "KAGGLE" and REHEARSE:
    from kgpu import export                       # noqa: E402

    # Two steps, one line each, in the same shape the RUN cell prints — `capture()` re-emits
    # `export`'s and `rehearse`'s own output as the DETAIL of the step that produced it.
    DRESS = progress.Stages([("export", "stage payload", 1.0),
                             ("rehearse", "rehearse worker", 1.0)], label=LABEL)
    DRESS.begin("export", "local, no upload, no quota")
    with DRESS.capture():
        export.export(CFG)                        # -> .payload/<job>/  (no upload)
    DRESS.begin("rehearse", "both Kaggle mount layouts")
    with DRESS.capture():
        runner.rehearse(CFG)
    DRESS.done("rehearsed — nothing was spent")
else:
    print("skipped" if ENVIRONMENT == "KAGGLE" else "LOCAL — nothing to rehearse")


LOCAL — nothing to rehearse


## 6 · Run


In [6]:
# ── RUN ───────────────────────────────────────────────────────────────────────
# ⚠️ One line shape on both machines — ` 33.7% - <task> - <sub-task> - <detail>`, one formatter
# (`utils.progress`), so the two cannot drift. LOCAL the task is the DOCUMENT and the sub-task
# its position in the cascade; KAGGLE the task is the STEP of the round trip.
# ⚠️ THE OVERALL % IS A POSITION IN THE PLAN, NOT A FRACTION OF THE TIME. A filing accepted at
# its FIRST OCR pass is ~1 min and one that defeats all 24 of them was 33, so the number is a
# LOWER BOUND — a run finishes early, it does not stall at 99 %. On KAGGLE it stands still
# through `wait kernel` unless this exact job has completed once before: `kernels_status`
# reports QUEUED / RUNNING / COMPLETE and no fraction.
# ⚠️ Each document's JSON is written BEFORE the next starts, and LOCAL each quarter is upserted
# as it lands — a run that kept its results in memory would lose them to the first interrupt.
# ⚠️ Budget: ~1 min for a filing accepted at layer 1, 26-33 min for one that defeats the whole
# cascade, plus Kaggle's ~5 min QUEUE before anything starts.
FOLDER = EXIT = REPORT = None

if not EXECUTE:
    print("EXECUTE = False — the plan above is resolved and nothing was spent")
elif ENVIRONMENT == "LOCAL":
    # `job.run` prints the progress line itself and writes the SAME line into the run
    # folder's `run.log`, so what you read here is what a later reader gets.
    FOLDER = job.run(SPEC)
else:
    if MERGE_INTO_CSV:
        print("MERGE_INTO_CSV is on: accepted statements are upserted into\n"
              "    raw_data/.../statements/ after the pull, with a backup taken first\n"
              "    and every changed cell printed.")
    # `refresh_data=True` re-exports and re-uploads the payload every time — correct, because
    # the filter above may have changed since the last run of this job.
    REPORT = progress.Stages(runner.RUN_STAGES, label=LABEL)
    EXIT = runner.run(CFG, refresh_data=True, progress=REPORT)
    REPORT.note(f"exit {EXIT}   (0 = COMPLETE and pulled)")


web_scraper.pdf_ocr_job
started   2026-09-02 03:25:35 GMT+7
device    cuda
gpu       NVIDIA GeForce RTX 3050 Laptop GPU  |  4,096 MiB total, 3,303 MiB free  |  CUDA 12.1, torch 2.5.1+cu121   [torch]
  0.0% - HOSE_TCB - plan - overall %    : documents finished + OCR passes done of the 7 this 55-layer cascade can cost, over 9 document(s) — a POSITION IN THE PLAN, never a fraction of the time
  0.0% - HOSE_TCB - plan - symbol       : HOSE_TCB
  0.0% - HOSE_TCB - plan - template     : bank   (detect_template (CafeF fingerprint, over the network))
  0.0% - HOSE_TCB - plan - documents    : 9  (Q4-2009, Q4-2010, Q2-2012, Q1-2013, Q1-2017, Q3-2017, Q2-2019, Q3-2019 …)
  0.0% - HOSE_TCB - plan - quarters     : all   selected ['2009-Q4', '2010-Q4', '2012-Q2', '2013-Q1', '2017-Q1', '2017-Q3', '2019-Q2', '2019-Q3', '2021-Q1']
  0.0% - HOSE_TCB - plan - skipped      : 50 quarter(s) already `pdf` in all three statements (2011-Q4, 2012-Q4, 2013-Q2, 2013-Q3, 2013-Q4, 2014-Q1, 2014-Q2, 2014-Q3 …)
  0.0

d:\GIT\master-thesis\mt_env\Lib\site-packages\gdown\__init__.py:3: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
d:\GIT\master-thesis\mt_env\Lib\site-packages\vietocr\tool\predictor.py:20: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explic

  0.6% - doc 1/9 HOSE_TCB Q4-2009 - layer 1/55 onnx@200 - page 2/5  40% of this pass
  1.0% - doc 1/9 HOSE_TCB Q4-2009 - layer 1/55 onnx@200 - page 3/5  60% of this pass   ~5 s left
  1.3% - doc 1/9 HOSE_TCB Q4-2009 - layer 1/55 onnx@200 - page 4/5  80% of this pass   ~2 s left
  1.6% - doc 1/9 HOSE_TCB Q4-2009 - layer 1/55 onnx@200 - page 5/5 100% of this pass   ~0 s left
  1.6% - doc 1/9 HOSE_TCB Q4-2009 - layer 2/55 onnx@300 - OCR pass 2/7
  1.9% - doc 1/9 HOSE_TCB Q4-2009 - layer 2/55 onnx@300 - page 1/5  20% of this pass
  2.2% - doc 1/9 HOSE_TCB Q4-2009 - layer 2/55 onnx@300 - page 2/5  40% of this pass
  2.5% - doc 1/9 HOSE_TCB Q4-2009 - layer 2/55 onnx@300 - page 3/5  60% of this pass   ~2 s left
  2.9% - doc 1/9 HOSE_TCB Q4-2009 - layer 2/55 onnx@300 - page 4/5  80% of this pass   ~1 s left
  3.2% - doc 1/9 HOSE_TCB Q4-2009 - layer 2/55 onnx@300 - page 5/5 100% of this pass   ~0 s left
  3.2% - doc 1/9 HOSE_TCB Q4-2009 - layer 3/55 onnx@400 - OCR pass 3/7
  3.5% - doc 1/9 HOSE

d:\GIT\master-thesis\mt_env\Lib\site-packages\vietocr\tool\translate.py:115: RuntimeWarning: invalid value encountered in divide
  char_probs = np.sum(char_probs, axis=-1)/(char_probs>0).sum(-1)


 23.4% - doc 3/9 HOSE_TCB Q2-2012 - layer 1/55 onnx@200 - page 36/48  75% of this pass   ~12 s left
 23.6% - doc 3/9 HOSE_TCB Q2-2012 - layer 1/55 onnx@200 - page 41/48  85% of this pass   ~7 s left
 23.6% - doc 3/9 HOSE_TCB Q2-2012 - layer 1/55 onnx@200 - page 42: text lines are vertical (25/26 boxes) — reading it at /Rotate 90
 23.7% - doc 3/9 HOSE_TCB Q2-2012 - layer 1/55 onnx@200 - page 44: text lines are vertical (26/26 boxes) — reading it at /Rotate 90
 23.7% - doc 3/9 HOSE_TCB Q2-2012 - layer 1/55 onnx@200 - page 46/48  95% of this pass   ~2 s left
 23.8% - doc 3/9 HOSE_TCB Q2-2012 - layer 1/55 onnx@200 - WARNING:     cash flow: the FX line printed no figure and its label rode onto the closing balance — 28,090,877,000,000 moved from `hdtc_vi_dieu_chinh_anh_huong_cua_thay_doi_ty_gia` to `hdtc_vii_tien_va_cac_khoan_tuong_duong_tien_tai_thoi_diem_cuoi_ky`, FX left empty
 23.8% - doc 3/9 HOSE_TCB Q2-2012 - layer 2/55 onnx@300 - OCR pass 2/7
 23.8% - doc 3/9 HOSE_TCB Q2-2012 - layer 

## 7 · The result — verdicts from the run folder

| verdict | what it says |
|---|---|
| `REPRODUCED` | every cell, the winning **layer**, the unit and `publish_date` match disk |
| `DIFFERS` | one of those moved — the run names which, with both figures |
| `absent in this run` | the cascade refused the statement; section 8 says why |
| `no such statement on any page of this filing` | the filing does **not contain** it and no layer can change that — a `BÁO CÁO TÀI CHÍNH TÓM TẮT` (Mẫu CBTT-03) carries a balance sheet and a four-line P&L and **no cash flow at all**. `missing` is correct and permanent (§5 rule 24) |
| `no pdf row on disk to compare against` | ⚠️ a **RECOVERY, not a reproduction** — nothing scored it |
| *(refused)* | a cumulative income statement is not scored against a row covering a different span |

⚠️ **READ THE FIRST REFUSAL, NOT THE LAST.** A cascade's final refusal names the hardest path
tried, not the blocking defect — `fx not mapped` sent this repo down a wrong diagnosis for two
days, and six of the seven quarters blamed on it then parsed at a **strict** layer with no FX
change at all (CLAUDE.md §6-2-duovicies).


In [7]:
# ── READ THE RUN FOLDER ─────────────────────────────────────────────────
# ⚠️ Read back from disk rather than from anything in memory, so this measures what a later
# reader would actually get. `metadata.json` already carries the whole scorecard in `results`.
import json                                          # noqa: E402

PATTERN = f"*__{EXCHANGE.lower()}_{SYMBOL.lower()}__pdf_ocr"
FOLDERS = sorted((REPO / "reports" / "pdf_ocr").glob(PATTERN), key=lambda p: p.name)
LATEST = FOLDERS[-1] if FOLDERS else None
META = MERGE = None

if LATEST is None:
    print(f"no run folder matching {PATTERN}")
else:
    META = json.loads((LATEST / "metadata.json").read_text(encoding="utf-8"))
    inputs, ocr = META.get("inputs", {}), META.get("environment", {}).get("ocr", {})
    SCHEMA = META.get("schema_version", 1)
    print(LATEST.name)
    print(f"  commit       : {META.get('git_commit')}")
    # ⚠️ §5 rule 2 at the artefact: an older run folder carries none of the fields below, and
    # printing `None` for them would read as a VALUE rather than as "this run predates the
    # field". `schema_version` is what tells the two apart.
    V2 = SCHEMA >= 2
    OLDER = "— (schema v1: this run predates the field)"
    print(f"  filter       : quarters={inputs.get('quarters')}  "
          f"periods={inputs.get('periods')}  "
          f"overwrite={inputs.get('overwrite') if V2 else OLDER}")
    print(f"  skipped      : {inputs.get('skipped_already_parsed') if V2 else OLDER}")
    print(f"  template     : {inputs.get('template')}  ({inputs.get('template_how')})")
    # ⚠️ THE TWO OCR HALVES FAIL INDEPENDENTLY — detection is onnxruntime, recognition is torch
    # — so "the GPU was used" is two questions. `ORT-1` is a green run that was half on the CPU
    # because onnxruntime ADVERTISED a provider the session then could not create.
    print(f"  detection    : {(ocr.get('det_providers') or ['?'])[0]}"
          f"   (onnxruntime {ocr.get('onnxruntime')})")
    print(f"  recognition  : {ocr.get('recognizer_device')}")
    print(f"  stack        : {ocr.get('stack_fingerprint')}"
          + (f"   ⚠️ PIN VIOLATIONS: {ocr['pin_violations']}"
             if ocr.get("pin_violations") else ""))

    # ⚠️ THE UPSERT IS THE ONE FIELD THE PARSING PROCESS CANNOT KNOW. `metadata.json` is written
    # by whatever ran the OCR, and on KAGGLE that is a worker with no path to this disk — so
    # `merged_into_csv` read `false` on every Kaggle run ever, whatever the pull did (`MRG-1`).
    # Since schema v3 the merge writes its own outcome back, and an ABSENT block on an older
    # folder means "this run predates the field", never "nothing was written".
    MERGE = META.get("merge")
    if MERGE:
        print(f"  upserted     : {MERGE['statements_written']} statement(s) written, "
              f"{MERGE['statements_skipped']} refused"
              + (f"   backup={inputs.get('merge_backup')}"
                 if inputs.get("merge_backup") else ""))
        if MERGE["periods_written"]:
            got = MERGE["periods_written"]
            print(f"                 {len(got)} quarter(s): "
                  f"{', '.join(got[:8])}{' …' if len(got) > 8 else ''}")
    elif SCHEMA >= 3:
        print("  upserted     : ⚠️ NOTHING — no merge ran against this run folder.")
    else:
        print(f"  upserted     : — (schema v{SCHEMA} predates the `merge` block; "
              f"`inputs.merged_into_csv` says {inputs.get('merged_into_csv')}, "
              f"which on a KAGGLE run was always false)")

    print()
    print(f"  {'period':10} {'report':18} {'layer':30} {'items':>5}  {'status':8} verdict")
    for r in META.get("results", []):
        print(f"  {r['period']:10} {r['report']:18} {(r['layer'] or '—'):30} "
              f"{r['items']:>5}  {r['status']:8} {r['verdict']}")
    # ⚠️ `seconds` is the DOCUMENT's cost repeated on each of its three report rows, so it is
    # summed per PERIOD. A set would also collapse two documents that took the same time.
    PER_DOC = {r["period"]: r["seconds"] for r in META.get("results", [])}
    print(chr(10) + f"  parse: {sum(PER_DOC.values()) / 60:.1f} min over "
          f"{len(PER_DOC)} document(s)")


20260902-032532__hose_tcb__pdf_ocr
  commit       : 4ee9daab
  filter       : quarters=None  periods=None  overwrite=False
  skipped      : ['Q4-2011', 'Q4-2012', 'Q2-2013', 'Q3-2013', 'Q4-2013', 'Q1-2014', 'Q2-2014', 'Q3-2014', 'Q4-2014', 'Q1-2015', 'Q2-2015', 'Q3-2015', 'Q4-2015', 'Q1-2016', 'Q2-2016', 'Q3-2016', 'Q4-2016', 'Q2-2017', 'Q4-2017', 'Q1-2018', 'Q2-2018', 'Q3-2018', 'Q4-2018', 'Q1-2019', 'Q4-2019', 'Q1-2020', 'Q2-2020', 'Q3-2020', 'Q4-2020', 'Q2-2021', 'Q3-2021', 'Q4-2021', 'Q1-2022', 'Q2-2022', 'Q3-2022', 'Q4-2022', 'Q1-2023', 'Q2-2023', 'Q3-2023', 'Q4-2023', 'Q1-2024', 'Q2-2024', 'Q3-2024', 'Q4-2024', 'Q1-2025', 'Q2-2025', 'Q3-2025', 'Q4-2025', 'Q1-2026', 'Q2-2026']
  template     : bank  (detect_template (CafeF fingerprint, over the network))
  detection    : CUDAExecutionProvider   (onnxruntime 1.22.0)
  recognition  : cuda
  stack        : e6b778e294b4
  upserted     : 0 statement(s) written, 27 refused

  period     report             layer                          

## 8 · Refused vs written — two questions, two places

The PARSE refused a statement → `run.log`, written by whatever ran the OCR. The MERGE refused
one → the `merge` block, written by whatever ran the upsert. ⚠️ **On KAGGLE those are two
machines**, and conflating them is how an 8-hour run came to look like a success while writing
0 of 201 accepted cells.


In [8]:
# ── WHAT WAS REFUSED, AND WHAT WAS WRITTEN ───────────────────────────────
#   the PARSE refused a statement   -> `run.log`, written by whatever ran the OCR
#   the MERGE refused a statement   -> the `merge` block, written by whatever ran the UPSERT
# ⚠️ ON KAGGLE THOSE ARE TWO MACHINES. A cell that greps the worker's `run.log` for
# `WRITE `/`skip ` finds nothing on a Kaggle run and, finding nothing, used to print "no
# refusals — every statement was accepted". That false success was printed over a run that
# wrote 0 of 201 accepted cells (HOSE_CTG, 2026-08-30).
# ⚠️ MATCH ON THE DETAIL, NOT ON THE START OF THE LINE: since 2026-08-30 every line reads
# ` xx.x% - task - sub-task - detail`, and `progress.detail_of` is the segment that used to BE
# the line.
if LATEST is not None:
    LOG = (LATEST / "run.log").read_text(encoding="utf-8", errors="replace")
    HITS = [ln for ln in LOG.splitlines()
            if "absent after" in ln or "reconcile:" in ln or "sane:" in ln]
    print("── the PARSE refused ────────────────────────────────────────")
    print(chr(10).join(HITS) if HITS else
          "  nothing — every statement the cascade opened was accepted")

    # ⚠️ NOT a refusal — a fact about the FILING. A page whose scan is turned reads as vertical
    # noise, and before 2026-08-30 that cost a whole statement in silence (BID Q3-2011's income
    # statement, `no such statement on any page of this filing`, 47 times).
    TURNED = [ln for ln in LOG.splitlines() if "text lines are vertical" in ln]
    if TURNED:
        print(chr(10) + "── pages the READ had to turn ──────────────────────────────")
        for ln in TURNED:
            print("  " + progress.detail_of(ln))

    print(chr(10) + "── the MERGE decided ───────────────────────────────────────")
    if MERGE:
        for ev in MERGE["events"]:
            for d in ev["decisions"]:
                mark = "WRITE " if d["action"] == "write" else "skip  "
                items = f"[{d['layer']}] {d['items']} items" if d["layer"] else ""
                print(f"  {mark} {d['period']:9} {d['report']:18} {items:32} {d['reason']}")
        print(chr(10) + f"  -> {MERGE['statements_written']} written, "
              f"{MERGE['statements_skipped']} refused")
        if not MERGE["statements_written"]:
            print("  ⚠️ NOTHING REACHED raw_data/. On a ticker with no CSV yet the "
                  "commonest reason is an")
            print("     EMPTY `sane` band — set FORCE_EMPTY_BAND = True and merge again. "
                  "It lifts ONE guard")
            print("     and no other, so screen the artefact before quoting anything "
                  "(`BND-1`).")
    else:
        # A LOCAL run before schema v3 wrote its merge decisions into `run.log` instead.
        OLD = [ln for ln in LOG.splitlines()
               if progress.detail_of(ln).startswith(
                   ("WRITE ", "skip ", "backup:", "written:"))]
        if OLD:
            print(chr(10).join(OLD))
        else:
            print("  ⚠️ NO MERGE RAN against this run folder — the statement CSVs "
                  "were not opened.")
            print("     KAGGLE: `python -m kgpu merge <job>` finishes it "
                  "(--force-empty-band for a")
            print("     ticker with no CSV yet).   LOCAL: set MERGE_INTO_CSV = True.")


── the PARSE refused ────────────────────────────────────────
 11.1% - doc 1/9 HOSE_TCB Q4-2009 - layer 55/55 onnx@200+joinlost+relax+components - WARNING:     cash_flow absent after 55 layer(s):
 22.2% - doc 2/9 HOSE_TCB Q4-2010 - layer 55/55 onnx@200+joinlost+relax+components - WARNING:     cash_flow absent after 55 layer(s):
 33.3% - doc 3/9 HOSE_TCB Q2-2012 - layer 55/55 onnx@200+joinlost+relax+components - WARNING:     balance_sheet absent after 55 layer(s):
 33.3% - doc 3/9 HOSE_TCB Q2-2012 - layer 55/55 onnx@200+joinlost+relax+components - WARNING:       [onnx@200] reconcile: no total assets
 33.3% - doc 3/9 HOSE_TCB Q2-2012 - layer 55/55 onnx@200+joinlost+relax+components - WARNING:       [tesseract@400+relax] reconcile: assets != liabilities + equity
 44.4% - doc 4/9 HOSE_TCB Q1-2013 - layer 55/55 onnx@200+joinlost+relax+components - WARNING:     balance_sheet absent after 55 layer(s):
 44.4% - doc 4/9 HOSE_TCB Q1-2013 - layer 55/55 onnx@200+joinlost+relax+components - WARNING

## 9 · Did it land? — the statement CSVs themselves

Everything above reports what some process DECIDED; this reads what is on disk.


In [9]:
# ── DID IT LAND? — the statement CSVs themselves ───────────────────────────
# ⚠️ Everything above reports what some process DECIDED; this reads what is on disk. The two
# came apart on a Kaggle round trip that finished green, wrote a complete run folder and created
# no CSV at all (`BND-1`, HOSE_BSR and again HOSE_CTG) — and no amount of log reading would have
# said so, because the merge that refused everything ran on the other machine.
import csv                                            # noqa: E402

from web_scraper import cafef_financials as fin       # noqa: E402

# ⚠️ `CWD-1`, AND THIS CELL WALKED STRAIGHT INTO IT. `statement_path()` reads
# `fin.STATEMENTS_DIR` at call time and its module default is RELATIVE — while the SETUP cell
# `os.chdir`s to `src/kaggle_gpu`, where `kgpu` stages its payload. So the first version of this
# cell reported `NO FILE` for a ticker whose three CSVs were on disk. The path is PRINTED,
# because a directory nobody names is a directory nobody checks.
ROOT = job.use_data_root(job.DEFAULT_DATA_ROOT)

TPL = (META or {}).get("inputs", {}).get("template") or TEMPLATE
# ⚠️ KEYED BY (period, REPORT), not by period. A merge writes one statement of a quarter and
# skips another — VCB Q1-2026 wrote its income statement and cash flow while its balance sheet
# was `identical to the row already on disk` — so a period-only set credits this run with a row
# it deliberately left alone.
MINE = {(d["period"], d["report"])
        for ev in (MERGE or {}).get("events", [])
        for d in ev["decisions"] if d["action"] == "write" and ev["applied"]}
if TPL is None:
    print("no template resolved — run the cells above first")
else:
    print(f"{ROOT / 'financials' / 'statements' / TPL}   {EXCHANGE}_{SYMBOL}")
    print()
    ANY = False
    for _report in fin.REPORTS:
        _path = Path(fin.statement_path(TPL, _report, EXCHANGE, SYMBOL))
        if not _path.is_file():
            print(f"  {_report:18} ⚠️ NO FILE — {_path.name} does not exist")
            continue
        ANY = True
        with open(_path, encoding="utf-8-sig") as _f:
            _rows = list(csv.DictReader(_f))
        _src = {}
        for _r in _rows:
            _src[_r.get("source", "")] = _src.get(_r.get("source", ""), 0) + 1
        _mine = [_r for _r in _rows if _r.get("source") == "pdf"
                 and (_r["period"], _report) in MINE]
        print(f"  {_report:18} {len(_rows):>3} quarters   "
              + "  ".join(f"{k}={v}" for k, v in sorted(_src.items()))
              + (f"   <- {len(_mine)} from this run" if _mine else ""))
        # ⚠️ Rule 24: a financial statement comes from the filing PDF and from nothing else.
        # A `cafef` row is an HTML transcription and must not be in this file.
        if _src.get("cafef"):
            print(f"       ⚠️ {_src['cafef']} row(s) read `source=cafef` — an HTML "
                  f"transcription. §5 rule 24 forbids it.")
    if not ANY:
        print()
        print("  ⚠️ THIS TICKER HAS NO STATEMENT CSV AT ALL. The parse is in the run folder "
              "and")
        print("     nothing was upserted — the MERGE section above says which refusal "
              "stopped it.")


D:\GIT\master-thesis\raw_data\cafef\financials\statements\bank   HOSE_TCB

  balance_sheet       67 quarters   missing=10  pdf=57
  income_statement    67 quarters   missing=8  pdf=59
  cash_flow           67 quarters   missing=15  pdf=52


## 10 · Repair one row — scoped, deliberate, read the diff first


In [10]:
# ── REPAIR ONE ROW — scoped, deliberate, and read the diff first ──────────
# ⚠️ THE ONLY WAY THIS NOTEBOOK OVERWRITES A GOOD-LOOKING `pdf` ROW. `pdf_ocr_merge` refuses a
# figure that DIFFERS from a `pdf` row on disk, because two runs disagreeing is not settled by
# preferring the newer one. That refusal is lifted here for the NAMED pairs only — never for the
# run — and `merge_run`'s own `periods`/`reports` filter is what scopes it.
# ⚠️ A BACKUP is taken before any write. Diff EVERY COLUMN afterwards, not the figures: three
# separate runs in this repo lost only a `publish_date` and a figures-only diff called each of
# them clean (CLAUDE.md §6-2-quatervicies, §6-2-quinvicies, §6-2-quadragies).
if LATEST is not None and REPAIR:
    from web_scraper import pdf_ocr_merge                 # noqa: E402

    HOW = "APPLY" if REPAIR_APPLY else "PLAN"
    print(f"{HOW} — {len(REPAIR)} scoped repair(s) from {LATEST.name}")
    print()
    for _period, _report in REPAIR:
        print(f"── {_period} {_report} " + "─" * 46)
        _rep = pdf_ocr_merge.merge_run(
            LATEST, apply=REPAIR_APPLY, periods=[_period], reports=[_report],
            force_differs=True, force_empty_band=FORCE_EMPTY_BAND)
        if getattr(_rep, "backup", None):
            print(f"   backup: {_rep.backup}")
    if not REPAIR_APPLY:
        print()
        print("nothing was written. Set REPAIR_APPLY = True to apply the plan above.")
elif LATEST is not None:
    print("REPAIR is empty — no row already on disk was replaced.")
    print("  A statement this run parsed that disk already holds as `pdf` was refused as")
    print("  DIFFERS and left alone. That is the default and usually right; name the")
    print("  (quarter, statement) pair in REPAIR only once the FILING has settled which")
    print("  reading is correct.")


REPAIR is empty — no row already on disk was replaced.
  A statement this run parsed that disk already holds as `pdf` was refused as
  DIFFERS and left alone. That is the default and usually right; name the
  (quarter, statement) pair in REPAIR only once the FILING has settled which
  reading is correct.
